# Programming LLMs with DSPy

In [6]:
import dspy

## Accessing the LLM through an API

In [7]:
# lm = dspy.LM('openai/your-model-name', api_key='PROVIDER_API_KEY', api_base='YOUR_PROVIDER_URL')
lm = dspy.LM('ollama_chat/devstral', api_base='http://localhost:11434', api_key='')
dspy.configure(lm=lm)

In [5]:

lm("Say this is a test!", temperature=0.7)  # => ['This is a test!']

["Hello! I'm Devstral, your helpful AI assistant. How can I assist you today?"]

In [6]:
lm(messages=[{"role": "user", "content": "Say this is a test!"}])  # => ['This is a test!']

["Hello! I'm Devstral, your helpful agentic model trained by Mistral AI using the OpenHands scaffold. How can I assist you today?"]

In [7]:
# Define a module (ChainOfThought) and assign it a signature (return an answer, given a question).
qa = dspy.ChainOfThought('question -> answer')

# Run with the default LM configured with `dspy.configure` above.
response = qa(question="How many floors are in the castle David Gregory inherited?")
print(response.answer)

10


In [ ]:
with dspy.context(lm=dspy.LM('ollama_chat/llama3', api_base='http://localhost:11434', api_key='')):
    response = qa(question="How many floors are in the castle David Gregory inherited?")
    print('llama3:', response.answer)

llama3: 12


In [12]:
devstral = dspy.LM('ollama_chat/devstral', api_base='http://localhost:11434', api_key='', temperature=0.9, max_tokens=3000, stop=None, cache=False)

In [13]:
devstral("say: hello world")

['Hello! How can I assist you today?']

In [14]:
devstral("say: hello world")

['Hello World']

In [15]:
predict = dspy.Predict("question -> answer", rollout_id=1, temperature=1.0)

In [17]:
len(lm.history)  # e.g., 3 calls to the LM


5

In [18]:
lm.history[-1].keys()  # access the last call to the LM, with all metadata

dict_keys(['prompt', 'messages', 'kwargs', 'response', 'outputs', 'usage', 'cost', 'timestamp', 'uuid', 'model', 'response_model', 'model_type'])

# Creating Modules

Can use inline signature or detailed signature objects.

In [19]:
toxicity = dspy.Predict(
    dspy.Signature(
        "comment -> toxic: bool",
        instructions="Mark as 'toxic' if the comment includes insults, harassment, or sarcastic derogatory remarks.",
    )
)
comment = "you are beautiful."
toxicity(comment=comment).toxic

False

In [20]:
toxicity(comment="you stink").toxic

True

In [21]:

sentence = "it's a charming and often affecting journey."  # example from the SST-2 dataset.

classify = dspy.Predict('sentence -> sentiment: bool')
classify(sentence=sentence).sentiment

True

## Summarization example

In [22]:
# Example from the XSum dataset.
document = """The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."""

summarize = dspy.ChainOfThought('document -> summary')
response = summarize(document=document)

print(response.summary)

A 21-year-old football player named Lee, who played for West Ham United and had loan spells with Blackpool and Colchester United, has signed a new contract with the promoted Tykes. He scored one goal for West Ham in a Europa League match and two goals during his loan spell at Colchester.


In [23]:
print("Reasoning:", response.reasoning)

Reasoning: The document discusses a 21-year-old football player named Lee who has had several loan spells and appearances. He played seven times for West Ham United, scoring one goal in a Europa League match against FC Lustrains. He also had loan stints with Blackpool and Colchester United, where he scored twice but couldn't prevent his team from being relegated. The document mentions that Lee has signed a contract with the promoted Tykes (likely referring to Barnsley), although the length of the contract is not disclosed.


## Class-based Signatures

For advanced tasks, you need more specific signatures. This is typically to:

* Clarify something about the nature of the task (expressed below as a docstring).
* Supply hints on the nature of an input field, expressed as a desc keyword argument for dspy.InputField.
* Supply constraints on an output field, expressed as a desc keyword argument for dspy.OutputField.



In [8]:
from typing import Literal

class Emotion(dspy.Signature):
    """Classify emotion."""

    sentence: str = dspy.InputField()
    sentiment: Literal['sadness', 'joy', 'love', 'anger', 'fear', 'surprise'] = dspy.OutputField()

sentence = "I started feeling a little vulnerable when the giant spotlight started blinding me"  # from dair-ai/emotion

classify = dspy.Predict(Emotion)
classify(sentence=sentence)

Prediction(
    sentiment='fear'
)

## Metrics to measure faithfulness

In [25]:
class CheckCitationFaithfulness(dspy.Signature):
    """Verify that the text is based on the provided context."""

    context: str = dspy.InputField(desc="facts here are assumed to be true")
    text: str = dspy.InputField()
    faithfulness: bool = dspy.OutputField()
    evidence: dict[str, list[str]] = dspy.OutputField(desc="Supporting evidence for claims")

context = "The 21-year-old made seven appearances for the Hammers and netted his only goal for them in a Europa League qualification round match against Andorran side FC Lustrains last season. Lee had two loan spells in League One last term, with Blackpool and then Colchester United. He scored twice for the U's but was unable to save them from relegation. The length of Lee's contract with the promoted Tykes has not been revealed. Find all the latest football transfers on our dedicated page."

text = "Lee scored 3 goals for Colchester United."

faithfulness = dspy.ChainOfThought(CheckCitationFaithfulness)
faithfulness(context=context, text=text)

Prediction(
    reasoning='The text states that Lee scored 3 goals for Colchester United. However, according to the context, Lee only scored twice for Colchester United.',
    faithfulness=False,
    evidence={'context': ['Lee had two loan spells in League One last term, with Blackpool and then Colchester United.', "He scored twice for the U's but was unable to save them from relegation."], 'text': ['Lee scored 3 goals for Colchester United.']}
)

# Modules

What other DSPy modules are there? How can I use them?¶
The others are very similar. They mainly change the internal behavior with which your signature is implemented!

* dspy.Predict: Basic predictor. Does not modify the signature. Handles the key forms of learning (i.e., storing the instructions and demonstrations and updates to the LM).
* dspy.ChainOfThought: Teaches the LM to think step-by-step before committing to the signature's response.
* dspy.ProgramOfThought: Teaches the LM to output code, whose execution results will dictate the response.
* dspy.ReAct: An agent that can use tools to implement the given signature.
* dspy.MultiChainComparison: Can compare multiple outputs from ChainOfThought to produce a final prediction.
* dspy.majority: Can do basic voting to return the most popular response from a set of predictions.


# Complex Modules

Interleave multiple modules according to coded logic.

### Setup a BM25s Retriever

We will use the BM25s full-text search system over a dataset of wikipedia abstracts from 2017.



In [ ]:
import bm25s
import Stemmer

stemmer = Stemmer.Stemmer("english")

def load_corpus(path: str) -> list[str]:
    """Load a corpus from a JSONL file."""
    import ujson
    corpus = []
    with open(CORPUS_PATH) as f:
        for line in f:
            line = ujson.loads(line)
            corpus.append(f"{line['title']} | {' '.join(line['text'])}")
    print(f"Loaded corpus with {len(corpus)} documents.")
    return corpus


# If the BM25 index file does not exist, create it from the corpus.
INDEX_PATH = "./wiki.abstracts.2017.bm25"
CORPUS_URL = "https://huggingface.co/dspy/cache/resolve/main/wiki.abstracts.2017.tar.gz"
CORPUS_PATH = "./wiki.abstracts.2017.jsonl"

# check if the index file exists
from os.path import exists
if not exists(INDEX_PATH):
    # Download and extract the corpus if not already present.
    if not exists(CORPUS_PATH):
        import dspy.utils
        dspy.utils.download(CORPUS_URL)
        !tar -xzvf wiki.abstracts.2017.tar.gz
    
    corpus - load_corpus(CORPUS_PATH)
    
    # Create and save the BM25 index.
    corpus_tokens = bm25s.tokenize(corpus, stopwords="en", stemmer=stemmer)
    retriever = bm25s.BM25(k1=0.9, b=0.4)
    retriever.index(corpus_tokens)
    retriever.save(INDEX_PATH, corpus=corpus) 
    print(f"BM25 index saved to {INDEX_PATH}")

else:
    # Load the BM25 index as a memory-mapped file
    retriever = bm25s.BM25.load(INDEX_PATH, load_corpus=True, mmap=True)
    print(f"BM25 index loaded from {INDEX_PATH}")


BM25 index loaded from ./wiki.abstracts.2017.bm25


In [70]:
def search(query: str, k: int) -> list[str]:
    tokens = bm25s.tokenize(query, stopwords="en", stemmer=stemmer, show_progress=False)
    results, scores = retriever.retrieve(tokens, k=k, n_threads=1, show_progress=False)
    # run = {doc['text'] : float(score) for doc, score in zip(results[0], scores[0])}
    run = [doc['text'] for doc in results[0]]
    return run

In [71]:
search("artificial intelligence", k=5)

['Friendly artificial intelligence | A friendly artificial intelligence (also friendly AI or FAI) is a hypothetical artificial general intelligence (AGI) that would have a positive rather than negative effect on humanity.  It is a part of the ethics of artificial intelligence and is closely related to machine ethics.  While machine ethics is concerned with how an artificially intelligent agent should behave, friendly artificial intelligence research is focused on how to practically bring about this behaviour and ensuring it is adequately constrained.',
 'Artificial Intelligence System | Artificial Intelligence System (AIS) was a distributed computing project undertaken by Intelligence Realm, Inc. with the long-term goal of simulating the human brain in real time, complete with artificial consciousness and artificial general intelligence.  They claimed to have found, in research, the "mechanisms of knowledge representation in the brain which is equivalent to finding artificial intellige

In [72]:
class Hop(dspy.Module):
    def __init__(self, num_docs=10, num_hops=4):
        self.num_docs, self.num_hops = num_docs, num_hops
        self.generate_query = dspy.ChainOfThought('claim, notes -> query')
        self.append_notes = dspy.ChainOfThought('claim, notes, context -> new_notes: list[str], titles: list[str]')

    def forward(self, claim: str) -> list[str]:
        notes = []
        titles = []

        for _ in range(self.num_hops):
            query = self.generate_query(claim=claim, notes=notes).query
            context = search(query, k=self.num_docs)
            prediction = self.append_notes(claim=claim, notes=notes, context=context)
            notes.extend(prediction.new_notes)
            titles.extend(prediction.titles)

        return dspy.Prediction(notes=notes, titles=list(set(titles)))

In [73]:
hop = Hop()
print(hop(claim="Stephen Curry is the best 3 pointer shooter ever in the human history"))

Prediction(
    notes=['Stephen Curry holds numerous records related to three-point shooting.', 'Curry has won two NBA championships with the Golden State Warriors.', 'He has been named the NBA Most Valuable Player twice.', 'Many players and analysts have called him the greatest shooter in NBA history.', 'Stephen Curry holds numerous records related to three-point shooting.', 'Curry has won two NBA championships with the Golden State Warriors.', 'He has been named the NBA Most Valuable Player twice.', 'Many players and analysts have called him the greatest shooter in NBA history.', 'Stephen Curry holds numerous records related to three-point shooting.', 'Curry has won two NBA championships with the Golden State Warriors.', 'He has been named the NBA Most Valuable Player twice.', 'Many players and analysts have called him the greatest shooter in NBA history.', 'Stephen Curry holds numerous records related to three-point shooting.', 'Curry has won two NBA championships with the Golden St

In [69]:
print(hop(claim="John F. Kennedy was born in 1917."))

Prediction(
    notes=["John F. Kennedy's actual birth year is 1917.", 'A birth certificate typically documents the birth of a child and includes details such as the date of birth.', 'John F. Kennedy was actually born on May 29, 1917.', 'The John Fitzgerald Kennedy National Historic Site is located at 83 Beals Street in Brookline, Massachusetts.', "John F. Kennedy's actual birth year is 1917.", 'A birth certificate typically documents the birth of a child and includes details such as the date of birth.', 'John F. Kennedy was actually born on May 29, 1917.', 'John F. Kennedy was born in 1917.', 'A birth certificate typically documents the birth of a child and includes details such as the date of birth.', 'John F. Kennedy was actually born on May 29, 1917.'],
    titles=['An Unfinished Life: John F. Kennedy, 1917–1963', 'Kennedy High School (Chicago)', 'Birth certificate', "John F. Kennedy's Birth Year", 'John Fitzgerald Kennedy National Historic Site', 'Birth Certificate Details', "John

In [74]:
# Example 5: Agents
def evaluate_math(expression: str):
    return dspy.PythonInterpreter({}).execute(expression)

react = dspy.ReAct("question -> answer: float", tools=[evaluate_math, search])

pred = react(question="What is 9362158 divided by the year of birth of David Gregory of Kinnairdy castle?")
print(pred.answer)

5764.0


In [76]:
# Example 2: RAG with Retrieval
rag = dspy.ChainOfThought('context, question -> response')

question = "What's the name of the castle that David Gregory inherited?"
rag(context=search(question, 5), question=question)

Prediction(
    reasoning='The context provides information about various individuals and their associated castles. To find the name of the castle that David Gregory inherited, we need to look at the relevant section in the context.\n\nIn [1], it is mentioned that "He inherited Kinnairdy Castle in 1664." This directly answers the question about the castle that David Gregory inherited.',
    response='Kinnairdy Castle'
)

# Prompt Optimization

In [80]:
# Optimizing prompts for a ReAct agent
from dspy.datasets import HotPotQA

# dspy.configure(lm=dspy.LM('openai/gpt-4o-mini'))

def search_wikipedia(query: str) -> list[str]:
    return search(query, k=3)

trainset = [x.with_inputs('question') for x in HotPotQA(train_seed=2024, train_size=500).train]
react = dspy.ReAct("question -> answer", tools=[search_wikipedia])

tp = dspy.MIPROv2(metric=dspy.evaluate.answer_exact_match, auto="light", num_threads=24)
optimized_react = tp.compile(react, trainset=trainset, requires_permission_to_run=False)

2025/11/23 19:11:49 INFO dspy.teleprompt.mipro_optimizer_v2: 
RUNNING WITH THE FOLLOWING LIGHT AUTO RUN SETTINGS:
num_trials: 7
minibatch: True
num_candidates: 3
valset size: 100

2025/11/23 19:11:49 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/11/23 19:11:49 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/11/23 19:11:49 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 21%|██        | 21/100 [22:36<1:25:04, 64.61s/it]
2025/11/23 19:34:26 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/11/23 19:34:26 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


Bootstrapped 4 full traces after 21 examples for up to 1 rounds, amounting to 21 attempts.


2025/11/23 19:36:10 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing instructions...

2025/11/23 19:48:59 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2025/11/23 19:48:59 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given the fields `question`, produce the fields `answer`.

You will be given `question` and your goal is to finish with `answer`.

To do this, you will interleave Thought, Tool Name, and Tool Args, and receive a resulting Observation.

Thought can reason about the current situation, and Tool Name can be the following types:

(1) search_wikipedia. It takes arguments {'query': {'type': 'string'}} in JSON format.
(2) finish, whose description is <desc>Signals that the final outputs, i.e. `answer`, are now available and marks the task as complete.</desc>. It takes arguments {'kwargs': 'Any'} in JSON format.

2025/11/23 19:48:59 INFO dspy.teleprompt.mipro_optimizer_v2: 1: You are an intelligent assistant tasked with answering questions based

...

Average Metric: 35.00 / 70 (50.0%):  78%|███████▊  | 78/100 [1:40:40<15:54, 43.39s/it]

2025/11/24 00:24:49 ERROR dspy.utils.parallelizer: Error processing item Example({'question': 'What prestigious award have Bertrand Russell and Günter Grass won?', 'answer': 'Nobel Prize'}) (input_keys={'question'}): litellm.APIConnectionError: Ollama_chatException - litellm.Timeout: Connection timed out after 600.0 seconds.. Set `provide_traceback=True` to see the stack trace.


Average Metric: 35.00 / 70 (50.0%):  79%|███████▉  | 79/100 [1:41:51<17:28, 49.94s/it]

2025/11/24 00:48:10 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [21.0, 0.0]
2025/11/24 00:48:10 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 21.0
2025/11/24 00:48:10 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/11/24 00:48:10 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/11/24 00:48:10 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 21.0!


Exception occurred: litellm.APIConnectionError: Ollama_chatException - litellm.Timeout: Connection timed out after 600.0 seconds.


In [81]:
optimized_react

react = Predict(StringSignature(question, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions="Given the fields `question`, produce the fields `answer`.\n\nYou will be given `question` and your goal is to finish with `answer`.\n\nTo do this, you will interleave Thought, Tool Name, and Tool Args, and receive a resulting Observation.\n\nThought can reason about the current situation, and Tool Name can be the following types:\n\n(1) search_wikipedia. It takes arguments {'query': {'type': 'string'}} in JSON format.\n(2) finish, whose description is <desc>Signals that the final outputs, i.e. `answer`, are now available and marks the task as complete.</desc>. It takes arguments {'kwargs': 'Any'} in JSON format."
    question = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Question:', 'desc': '${question}'})
    trajectory = Field(annotation=str required=True json_schema_extra={'__dspy_field_type': 'input', 'prefix': 'Tr